In [1]:
data = 'covid_data.csv'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [3]:
df = pd.read_csv(data)

In [4]:
# Task 1

In [5]:
df['date'] = pd.to_datetime(df['date'])

In [6]:
cols = ['new_cases', 'date', 'location']
df[cols].isnull().sum()

new_cases    19276
date             0
location         0
dtype: int64

In [7]:
df['new_cases'] = df['new_cases'].fillna(0)

In [8]:
df['high_cases'] = (df['new_cases'] > 1000).astype(int)

In [9]:
df.columns

Index(['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases',
       'new_cases_smoothed', 'total_deaths', 'new_deaths',
       'new_deaths_smoothed', 'total_cases_per_million',
       'new_cases_per_million', 'new_cases_smoothed_per_million',
       'total_deaths_per_million', 'new_deaths_per_million',
       'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients',
       'icu_patients_per_million', 'hosp_patients',
       'hosp_patients_per_million', 'weekly_icu_admissions',
       'weekly_icu_admissions_per_million', 'weekly_hosp_admissions',
       'weekly_hosp_admissions_per_million', 'total_tests', 'new_tests',
       'total_tests_per_thousand', 'new_tests_per_thousand',
       'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
       'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations',
       'people_vaccinated', 'people_fully_vaccinated', 'total_boosters',
       'new_vaccinations', 'new_vaccinations_smoothed',
       't

In [10]:
df = df[['continent', 'iso_code', 'total_cases', 'total_deaths', 'total_vaccinations', 'population', 'new_cases', 'high_cases']]
df = df.dropna()

In [11]:
df

,continent,iso_code,total_cases,total_deaths,total_vaccinations,population,new_cases,high_cases
414,Asia,AFG,55604.0,2432.0,0.0,41128772,0.0,0
420,Asia,AFG,55714.0,2443.0,8200.0,41128772,110.0,0
436,Asia,AFG,55985.0,2457.0,54000.0,41128772,0.0,0
458,Asia,AFG,56676.0,2497.0,120000.0,41128772,0.0,0
473,Asia,AFG,57793.0,2539.0,240000.0,41128772,0.0,0
...,...,...,...,...,...,...,...,...
428761,Africa,ZWE,257340.0,5599.0,12212594.0,16320539,0.0,0
428762,Africa,ZWE,257517.0,5602.0,12214870.0,16320539,177.0,0
428763,Africa,ZWE,257517.0,5602.0,12216848.0,16320539,0.0,0
428765,Africa,ZWE,257517.0,5602.0,12219760.0,16320539,0.0,0


In [12]:
X = df.drop(['high_cases', 'new_cases'], axis=1)
y = df['high_cases']

In [13]:
# Task 2

In [14]:
X = pd.get_dummies(X, columns=['continent'], drop_first=True)

In [15]:
le = LabelEncoder()
X['iso_code'] = le.fit_transform(X['iso_code'])

In [16]:
scaler = StandardScaler()
num_columns = ['total_cases', 'total_deaths', 'total_vaccinations', 'population']
X[num_columns] = scaler.fit_transform(X[num_columns])

In [17]:
# Task 3

In [18]:
from imblearn.over_sampling import SMOTE

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=45)

In [20]:
smote = SMOTE(random_state=45)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [21]:
# Task 5

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [23]:
models = {
    "LR": LogisticRegression(max_iter=1000),
    "DT": DecisionTreeClassifier(),
    "RF": RandomForestClassifier(),
    "KNN": KNeighborsClassifier()
}

results = {}

In [24]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }
    print(f"\n{name}")
    print(confusion_matrix(y_test, y_pred))


LR
[[3385 8298]
 [ 241 1019]]

DT
[[10428  1255]
 [ 1019   241]]

RF
[[10669  1014]
 [ 1129   131]]

KNN
[[9875 1808]
 [ 974  286]]


In [25]:
pd.DataFrame(results).T

,Accuracy,Precision,Recall,F1
LR,0.340261,0.109370,0.808730,0.192682
DT,0.824307,0.161096,0.191270,0.174891
RF,0.834428,0.114410,0.103968,0.108940
KNN,0.785058,0.136581,0.226984,0.170543


In [26]:
# Task 6

In [27]:
from sklearn.feature_selection import SelectKBest, f_classif

In [28]:
selector = SelectKBest(score_func=f_classif, k=10)
X_new = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

In [29]:
selected_features

Index(['iso_code', 'total_cases', 'total_deaths', 'total_vaccinations',
       'population', 'continent_Asia', 'continent_Europe',
       'continent_North America', 'continent_Oceania',
       'continent_South America'],
      dtype='object')

In [30]:
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X[selected_features], y, test_size=0.2, random_state=45
    )

In [31]:
model = RandomForestClassifier()
model.fit(X_train_new, y_train_new)

y_pred = model.predict(X_test_new)

In [32]:
print(f"Accuracy: {accuracy_score(y_test_new, y_pred)}")

Accuracy: 0.854825001931546


In [33]:
# Task 4

In [34]:
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures

In [35]:

X_reg = df.drop(['new_cases', 'high_cases'], axis=1)
y_reg = df['new_cases']


X_reg = pd.get_dummies(X_reg, columns=['continent'], drop_first=True)
le_reg = LabelEncoder()
X_reg['iso_code'] = le_reg.fit_transform(X_reg['iso_code'])


scaler_reg = StandardScaler()
num_cols_reg = ['total_cases', 'total_deaths', 'total_vaccinations', 'population']
X_reg[num_cols_reg] = scaler_reg.fit_transform(X_reg[num_cols_reg])

print("X_reg shape:", X_reg.shape)
print("Features:", X_reg.columns.tolist())

X_reg shape: (64712, 10)
Features: ['iso_code', 'total_cases', 'total_deaths', 'total_vaccinations', 'population', 'continent_Asia', 'continent_Europe', 'continent_North America', 'continent_Oceania', 'continent_South America']


In [36]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=45
)

In [37]:
reg_models = {
    "Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(),
    "Ridge Regression": Ridge(),
}

reg_results = {}

In [38]:
for name, model in reg_models.items():
    model.fit(X_train_reg, y_train_reg)
    y_pred_reg = model.predict(X_test_reg)

    reg_results[name] = {
        "MSE": mean_squared_error(y_test_reg, y_pred_reg),
        "R2": r2_score(y_test_reg, y_pred_reg)
    }

pd.DataFrame(reg_results).T

,MSE,R2
Linear Regression,9.128279e+09,0.009945
Lasso Regression,9.128131e+09,0.009961
Ridge Regression,9.128197e+09,0.009954


In [39]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_reg)

X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X_poly, y_reg, test_size=0.2, random_state=45
)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train_poly)
y_pred_poly = poly_model.predict(X_test_poly)


poly_results = {
    "Polynomial Regression (degree=2)": {
        "MSE": mean_squared_error(y_test_poly, y_pred_poly),
        "R2": r2_score(y_test_poly, y_pred_poly)
    }
}

print("Polynomial Regression Results:")
pd.DataFrame(poly_results).T

Polynomial Regression Results:


,MSE,R2
Polynomial Regression (degree=2),1.396262e+10,-0.514389


In [40]:
all_results = {**reg_results, **poly_results}
comparison_df = pd.DataFrame(all_results).T
comparison_df = comparison_df.sort_values("R2", ascending=False)

print("All Regression Models Comparison (sorted by R²):")
comparison_df

All Regression Models Comparison (sorted by R²):


,MSE,R2
Lasso Regression,9.128131e+09,0.009961
Ridge Regression,9.128197e+09,0.009954
Linear Regression,9.128279e+09,0.009945
Polynomial Regression (degree=2),1.396262e+10,-0.514389
